In [61]:
import os
import torch
import torchaudio
import soundfile as sf
import pandas as pd
import numpy as np
import evaluate
from pathlib import Path
from tqdm.auto import tqdm
from typing import Dict, List, Union, Any
from dataclasses import dataclass
from datasets import DatasetDict, Dataset, Audio as DatasetsAudio
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from audiomentations import (
    Compose,
    AddGaussianNoise,
    AddBackgroundNoise, 
    ApplyImpulseResponse,
    Gain,
    BitCrush
)

In [62]:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
def patched_load(uri, **kwargs):
    data, sr = sf.read(uri)
    wav = torch.tensor(data).float()
    return (wav.unsqueeze(0) if wav.ndim == 1 else wav.T), sr
torchaudio.load = patched_load

In [63]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
local_file = 'v4_ru.pt'

if not os.path.isfile(local_file):
    import urllib.request
    url = 'https://models.silero.ai/models/tts/ru/v4_ru.pt'
    urllib.request.urlretrieve(url, local_file)

model = torch.package.PackageImporter(local_file).load_pickle("tts_models", "model")
model.to(device)

In [33]:
speakers = ['aidar', 'baya', 'kseniya', 'xenia', 'eugene']
sample_rate = 48000

example_text = "особенно пока я листаю Авито"
audio = model.apply_tts(text=example_text,
                        speaker='baya',
                        sample_rate=sample_rate)

import IPython.display as ipd
ipd.display(ipd.Audio(audio, rate=sample_rate))

In [43]:
phrases = [
    "авито поможет продать вещи", "ищу новую работу на авито", "посмотри это объявление на авито",
    "авито доставка очень удобная", "купил отличный диван на авито", "на авито всегда много предложений",
    "привет это поддержка авито", "разместил резюме на авито", "авито это сервис номер один",
    "нашел квартиру на авито за час", "на авито можно торговаться", "проверь свой профиль на авито",
    "авито объединяет людей", "продам старый ноутбук на авито", "на авито есть отзывы о продавцах",
    "отправлю товар через авито", "авито работает по всей россии", "ищи услуги мастеров на авито",
    "авито делает жизнь проще", "заходи на авито каждый день"
]

In [65]:
current_path = Path(".").resolve()
if current_path.name == "src":
    ROOT_DIR = current_path.parent
else:
    ROOT_DIR = current_path
AUDIO_DIR = ROOT_DIR / "data" / "audio"
DATASET_PATH = ROOT_DIR / "data" / "datasets" / "avito_samples"
FINAL_AUDIO_PATH = DATASET_PATH / "audio"
MODEL_NAME = "openai/whisper-small"



In [56]:
from audiomentations import (
    Compose,
    AddGaussianNoise,
    AddBackgroundNoise, 
    ApplyImpulseResponse,
)
from audiomentations import Gain, BitCrush


augs = Compose([
    AddBackgroundNoise(str(AUDIO_DIR / "noise_background_bubble.wav"), min_snr_db=10.0, max_snr_db=20.0, p=1.0),
    ApplyImpulseResponse(str(AUDIO_DIR / "rir_stairway.wav"), p=0.4),
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.01, p=0.5),
    Gain(min_gain_db=-6, max_gain_db=6, p=0.5),
    BitCrush(min_bit_depth=6, max_bit_depth=12, p=0.3)
])


In [57]:
records = []

for p_idx, phrase in enumerate(tqdm(phrases)):
    for speaker in speakers:
        audio = model.apply_tts(text=phrase, speaker=speaker, sample_rate=48000)
        for v_idx in range(3):
            file_id = f"avito_{p_idx}_{speaker}_v{v_idx}"
            aug_audio = augs(audio.numpy(), sample_rate=48000)
            aug_audio = torch.from_numpy(aug_audio).unsqueeze(0)
            resampler = torchaudio.transforms.Resample(orig_freq=48000, new_freq=8000)
            audio_8k = resampler(aug_audio)
            
            save_path = FINAL_AUDIO_PATH / f"{file_id}.wav"
            sf.write(save_path, audio_8k.squeeze().numpy(), 8000)
            records.append({"audio_id": file_id, "text": phrase})

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
df = pd.DataFrame(records).sample(frac=1, random_state=42)
split_idx = int(len(df) * 0.8)
df[:split_idx].to_csv(DATASET_PATH / "train.csv", index=False)
df[split_idx:].to_csv(DATASET_PATH / "test.csv", index=False)

In [68]:
def extract_dataset_dict_from_directory(dataset_dir: Path, csv_name: str) -> Dict[str, List[str]]:
    df = pd.read_csv(dataset_dir / csv_name)
    df = df.head(100)
    output = {"audio": [], "text": []}

    # Проходимся по всем строкам в файле разметки и сохраняем 1 - путь до аудио, 2 - текст из аудио
    for _, row in df.iterrows():
        output["audio"].append(str(dataset_dir / "audio" / f"{row.audio_id}.wav"))
        output["text"].append(row.text)
    
    return output

In [77]:
asr_dataset = DatasetDict({
    "train": Dataset.from_dict(extract_dataset_dict_from_directory(DATASET_PATH, "train.csv")),
    "test": Dataset.from_dict(extract_dataset_dict_from_directory(DATASET_PATH, "test.csv")),
})

In [72]:
# Создаем компоненты модели с целью их применения при подготовке датасета и обучении модели
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language="russian", task="transcribe")

In [73]:
# Подгружаем саму модель и настраиваем ее
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [78]:
# Описываем логику подготовки датасета
def prepare_dataset(batch):
    audio = batch["audio"]
    wav, sr = torchaudio.load(audio)
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        wav = resampler(wav)
    audio_array = wav.squeeze().numpy()

    # Извлекаем признаки. У Whisper -- это 64 мел-коэффициента
    batch["input_features"] = feature_extractor(
        audio_array,
        sampling_rate=16000
    ).input_features[0]
    
    # Токенизируем
    batch["labels"] = tokenizer(batch["text"]).input_ids
    return batch


# Применяем описанную функцию подготовки к датасету.
# Удаляем колонки audio и text -- они нам больше не понадобятся. 
# Вместо них держим input_features (мел-спектры) и labels (токены текстовок)
vectorized_asr_dataset = asr_dataset.map(
    prepare_dataset,
    remove_columns=asr_dataset["train"].column_names,
)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

In [79]:
# Опишем и инициализируем collate-класс (to collate - "сопоставлять")

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    feature_extractor: Any
    tokenizer: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch
    

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
    decoder_start_token_id=model.config.decoder_start_token_id
)

In [83]:
# Описываем параметры обучения
training_args = Seq2SeqTrainingArguments(
    # Здесь будут лежать логи и чекпоинты модели, появляющиеся по мере обучения
    output_dir="./finetuning-logs",

    num_train_epochs=15, 
    max_steps=-1,

    # Аргументы, связанные с процессом оптимизации
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_steps=50,
    #max_steps=4000,
    gradient_checkpointing=True,
    optim="adamw_torch",
    fp16=True,
    
    # Аргументы, связанные с генерацией транскрипции
    predict_with_generate=True,
    generation_max_length=225,

    # Аргументы, связанные с эвал-шагом
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    save_steps=50,
    eval_steps=50,

    # Аргументы, связанные с логгированием
    logging_steps=10,
    report_to=["tensorboard"],
    
    # Аргументы, связанные с оценкой метриками и отбором моделей
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

In [85]:
%pip install jiwer

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 5.6 MB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.5 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 3.7 MB/s  0:00:00

   -------------------- ------------------- 1/2 [jiwer]
   ---------------------------------------- 2/2 [jiwer]

Note: you may need to restart the kernel to use updated packages.


In [86]:
# Определяем функционал подсчета метрик
metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    
    # Replace -100 with pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    
    # Decode predictions and labels
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [88]:
# Подготавливаем объект-trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=vectorized_asr_dataset["train"],
    eval_dataset=vectorized_asr_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [89]:
# Запускаем обучение
trainer.train()

Step,Training Loss,Validation Loss,Wer
50,0.003007,0.010282,0.735294
60,0.000477,0.009173,0.735294


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its para

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=60, training_loss=1.6572319987462834, metrics={'train_runtime': 370.1433, 'train_samples_per_second': 4.052, 'train_steps_per_second': 0.162, 'total_flos': 4.3287810048e+17, 'train_loss': 1.6572319987462834, 'epoch': 15.0})

In [96]:
# Сохраняем модель
model.save_pretrained("D:/QT projects/whisper-finetuned")
processor = WhisperProcessor(feature_extractor, tokenizer)
processor.save_pretrained("D:/QT projects/whisper-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['D:/QT projects/whisper-finetuned\\processor_config.json']

In [97]:
device = "cuda:0"

model_type = "finetuned"
# model_type = "pretrained"

if model_type == "finetuned":
    model = WhisperForConditionalGeneration.from_pretrained("D:/QT projects/whisper-finetuned")
    model.to(device)
    processor = WhisperProcessor.from_pretrained("D:/QT projects/whisper-finetuned")

elif model_type == "pretrained":
    # Для сравнения можем применить не дообученную модель
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
    model.to(device)
    processor = WhisperProcessor.from_pretrained(MODEL_NAME)

else:
    raise ValueError("You've passed wrong `model_type`.")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [98]:
from jiwer import visualize_alignment, process_words


def postprocess_text(text: str) -> str:
    text = text.lower().strip()
    text = "".join(s for s in text if str(s).isalpha() or s == " ")

    return text


def infer_whisper(wav: torch.Tensor, sample_rate: int, lang: str = "en") -> str:
    # Обрабатываем аудио
    inputs = processor(
        wav.squeeze().numpy(),
        sampling_rate=sample_rate,
        return_tensors="pt",
    )

    # Генерируем транскрипцию
    outputs = model.generate(
        **inputs.to(device),
        language=f"<|{lang}|>"
    )

    hypothesis = processor.batch_decode(outputs, skip_special_tokens=True)[0]

    return hypothesis

In [99]:
# Читаем и выводим аудио
wav, sr = torchaudio.load("C:/Users/dkhan/Downloads/audio_seminar/data/audio/hello_avito.wav")
print("sample_rate:", sr)
display(Audio(data=wav, rate=sr))

# Ресэмплим, т.к. Whisper обучен на аудио в 16 кГц
wav = torchaudio.transforms.Resample(sr, 16000)(wav)

# Получаем транскрипцию
transcription = infer_whisper(wav, sample_rate=16000, lang="ru")
postprocess_text(transcription)

sample_rate: 8000


'вас приветствует компания авито'